In [0]:
# Retail Sales Lakehouse - Silver Layer
# Load Bronze Delta tables

customers_bronze = spark.table("workspace.default.bronze_customers")
products_bronze = spark.table("workspace.default.bronze_products")
orders_bronze = spark.table("workspace.default.bronze_orders")
order_items_bronze = spark.table("workspace.default.bronze_order_items")

print("Bronze tables loaded successfully")

print("Customers:", customers_bronze.count())
print("Products:", products_bronze.count())
print("Orders:", orders_bronze.count())
print("Order Items:", order_items_bronze.count())

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_check_customers = customers_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in customers_bronze.columns
])

display(null_check_customers)

In [0]:
duplicate_customers = (
    customers_bronze
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_customers)

In [0]:
invalid_products = products_bronze.filter(
    (col("unit_price").isNull()) | (col("unit_price") <= 0)
)

display(invalid_products)

In [0]:
invalid_order_items = order_items_bronze.filter(
    (col("quantity").isNull()) | (col("quantity") <= 0)
)

display(invalid_order_items)

In [0]:
valid_status = ["Completed", "Cancelled", "Pending"]
valid_channel = ["Online", "Store"]

invalid_orders = orders_bronze.filter(
    (~col("order_status").isin(valid_status)) |
    (~col("sales_channel").isin(valid_channel)) |
    col("order_id").isNull() |
    col("customer_id").isNull()
)

display(invalid_orders)

In [0]:
from pyspark.sql.functions import trim, lower, initcap

customers_silver = (
    customers_bronze
    .dropDuplicates(["customer_id"])
    .filter(col("customer_id").isNotNull())
    .withColumn("customer_name", initcap(trim(col("customer_name"))))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("city", initcap(trim(col("city"))))
    .withColumn("country", trim(col("country")))
)

display(customers_silver)

In [0]:
customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_customers")

In [0]:
display(spark.table("workspace.default.silver_customers"))

In [0]:
products_silver = (
    products_bronze
    .dropDuplicates(["product_id"])
    .filter(
        col("product_id").isNotNull() &
        col("unit_price").isNotNull() &
        (col("unit_price") > 0)
    )
    .withColumn("product_name", initcap(trim(col("product_name"))))
    .withColumn("category", initcap(trim(col("category"))))
    .withColumn("subcategory", initcap(trim(col("subcategory"))))
)

display(products_silver)

In [0]:
products_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_products")

In [0]:
orders_silver = (
    orders_bronze
    .dropDuplicates(["order_id"])
    .filter(
        col("order_id").isNotNull() &
        col("customer_id").isNotNull() &
        col("order_status").isin(["Completed", "Cancelled", "Pending"]) &
        col("sales_channel").isin(["Online", "Store"])
    )
    .withColumn("order_status", initcap(trim(col("order_status"))))
    .withColumn("sales_channel", initcap(trim(col("sales_channel"))))
)

display(orders_silver)

In [0]:
orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_orders")

In [0]:
order_items_silver = (
    order_items_bronze
    .dropDuplicates(["order_item_id"])
    .filter(
        col("order_item_id").isNotNull() &
        col("order_id").isNotNull() &
        col("product_id").isNotNull() &
        col("quantity").isNotNull() &
        (col("quantity") > 0)
    )
)

display(order_items_silver)

In [0]:
order_items_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_order_items")

In [0]:
silver_tables = [
    "workspace.default.silver_customers",
    "workspace.default.silver_products",
    "workspace.default.silver_orders",
    "workspace.default.silver_order_items"
]

for table in silver_tables:
    df = spark.table(table)
    print(f"{table} -> {df.count()} rows")